In [0]:
-- Q: which outlets earn the most?
CREATE OR REPLACE TABLE workspace.default.gold_sales_by_outlet AS
SELECT o.outlet_id, o.outlet_type, o.location_tier, o.outlet_size,
       2026 - o.established_year AS outlet_age,
       COUNT(*)                  AS items_sold,
       ROUND(SUM(f.sales), 0)    AS total_sales
FROM workspace.default.silver_fact_sales f
JOIN workspace.default.silver_dim_outlet o USING (outlet_id)
GROUP BY ALL;

-- Q: which product categories earn the most?
CREATE OR REPLACE TABLE workspace.default.gold_sales_by_category AS
SELECT p.product_type, p.fat_content,
       ROUND(SUM(f.sales), 0)  AS total_sales,
       ROUND(AVG(f.price), 2)  AS avg_price,
       RANK() OVER (ORDER BY SUM(f.sales) DESC) AS sales_rank
FROM workspace.default.silver_fact_sales f
JOIN workspace.default.silver_dim_product p USING (product_id)
GROUP BY p.product_type, p.fat_content;

-- Q: top 20 products
CREATE OR REPLACE TABLE workspace.default.gold_top_products AS
SELECT f.product_id, p.product_type,
       ROUND(SUM(f.sales), 0)        AS total_sales,
       COUNT(DISTINCT f.outlet_id)   AS outlets_sold_in
FROM workspace.default.silver_fact_sales f
JOIN workspace.default.silver_dim_product p USING (product_id)
GROUP BY ALL
ORDER BY total_sales DESC
LIMIT 20;

SELECT 'outlet' AS tbl, COUNT(*) AS n, SUM(total_sales) AS total FROM workspace.default.gold_sales_by_outlet
UNION ALL
SELECT 'category', COUNT(*), SUM(total_sales) FROM workspace.default.gold_sales_by_category;
SELECT ROUND(SUM(sales), 2) AS true_total
FROM workspace.default.silver_fact_sales;